# Process Raw Chamber Data With Plain Python

This notebook has no widget interface. Edit the variables in the configuration cell, then run the cells top to bottom.


## Setup


In [ ]:
import pathlib
import sys
import urllib.parse

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print

CWD = pathlib.Path.cwd().resolve()
for candidate in [CWD, *CWD.parents]:
    if (candidate / "pyproject.toml").exists():
        REPO_ROOT = candidate
        break
else:
    REPO_ROOT = CWD

NOTEBOOK_DIR = REPO_ROOT / "notebooks" / "processing"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from soilgasflux_fcs import Multiprocessor, json_reader

DEFAULT_OUTPUT_DIR = NOTEBOOK_DIR / "output"
DEFAULT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CANONICAL_COLUMNS = [
    "datetime",
    "id",
    "timedelta",
    "k30_co2",
    "si_temperature",
    "si_humidity",
    "bmp_pressure",
]


## Loading And Normalization Helpers


In [ ]:
def sanitize_input_path(input_path):
    text = str(input_path or "").strip()
    if not text:
        raise FileNotFoundError("Input path is empty. Paste a file or folder path before loading.")

    if text.startswith(("Path(", "pathlib.Path(")) and text.endswith(")"):
        text = text[text.find("(") + 1:-1].strip()

    for _ in range(2):
        if len(text) >= 2 and text[0] == text[-1] and text[0] in {"'", '"'}:
            text = text[1:-1].strip()

    if text.endswith("()"):
        text = text[:-2].strip()

    parsed = urllib.parse.urlparse(text)
    if parsed.scheme == "file":
        text = urllib.parse.unquote(parsed.path)

    if text.startswith("Users/"):
        text = "/" + text

    return pathlib.Path(text).expanduser()


def find_input_files(input_path, pattern="*.csv"):
    path = sanitize_input_path(input_path)
    if path.is_file():
        return [path]
    if path.is_dir():
        files = sorted(file for file in path.glob(pattern) if file.is_file())
        if files:
            return files
        raise FileNotFoundError(
            f"No CSV files matched pattern {pattern!r} in {path}. "
            "Check the input mode, folder, and CSV pattern."
        )
    raise FileNotFoundError(f"Input path does not exist: {path}")


def find_json_files(folder_path, min_size_bytes=5000):
    folder = sanitize_input_path(folder_path)
    if not folder.exists():
        raise FileNotFoundError(f"JSON folder does not exist: {folder}")
    if not folder.is_dir():
        raise NotADirectoryError(f"JSON input must be a folder, not a file: {folder}")

    all_json = sorted(file for file in folder.rglob("*.json") if file.is_file())
    usable = [file for file in all_json if file.stat().st_size >= min_size_bytes]
    ignored = [file for file in all_json if file.stat().st_size < min_size_bytes]

    if not all_json:
        raise FileNotFoundError(f"No JSON files found in this folder: {folder}")
    if not usable:
        raise FileNotFoundError(
            f"No usable JSON files found in {folder}. Found {len(all_json)} JSON file(s), "
            f"but all were smaller than {min_size_bytes} bytes."
        )
    return usable, ignored


def load_json_folder(folder_path, min_size_bytes=5000):
    folder = sanitize_input_path(folder_path)
    files, ignored = find_json_files(folder, min_size_bytes=min_size_bytes)
    initializer = json_reader.Initializer(folder)
    try:
        df = initializer.prepare_rawdata()
    except ValueError as exc:
        if "No objects to concatenate" in str(exc) or "no objects to concatenate" in str(exc):
            raise FileNotFoundError(
                f"No usable JSON raw-data files were loaded from {folder}. "
                "Check that the folder contains FCS JSON files with a raw_data section."
            ) from exc
        raise
    if df.empty:
        raise FileNotFoundError(f"JSON files were found in {folder}, but no rows were loaded.")
    df.attrs["source_path"] = str(folder)
    df.attrs["source_files"] = [str(file) for file in files]
    df.attrs["ignored_small_json_files"] = [str(file) for file in ignored]
    return df[CANONICAL_COLUMNS + [c for c in df.columns if c not in CANONICAL_COLUMNS]]


def load_csv_files(input_path, pattern="*.csv", delimiter=","):
    files = find_input_files(input_path, pattern=pattern)
    frames = []
    for file in files:
        df = pd.read_csv(file, sep=delimiter)
        df["__source_file"] = file.stem
        frames.append(df)
    if not frames:
        raise FileNotFoundError(
            f"No CSV files could be read from {sanitize_input_path(input_path)} with pattern {pattern!r}."
        )
    raw = pd.concat(frames, ignore_index=True)
    raw.attrs["source_files"] = [str(file) for file in files]
    return raw


def guess_column(columns, candidates):
    normalized = {str(col).lower().strip(): col for col in columns}
    for candidate in candidates:
        key = candidate.lower().strip()
        if key in normalized:
            return normalized[key]
    for col in columns:
        lower = str(col).lower()
        if any(candidate.lower() in lower for candidate in candidates):
            return col
    return ""


def _require_or_fill(raw_df, source_col, output_col, fill_missing_environment, default_value):
    if source_col:
        return raw_df[source_col]
    if fill_missing_environment:
        return default_value
    raise ValueError(
        f"Missing mapping for {output_col}. Select a source column or enable constant-fill mode."
    )


def normalize_csv_dataframe(
    raw_df,
    *,
    co2_col,
    timestamp_col="",
    elapsed_col="",
    id_col="",
    pressure_col="",
    temperature_col="",
    humidity_col="",
    pressure_unit="Pa",
    fill_missing_environment=False,
    default_pressure_pa=101325.0,
    default_temperature_c=20.0,
    default_humidity_percent=70.0,
):
    if not co2_col:
        raise ValueError("CO2 column is required.")
    if not timestamp_col and not elapsed_col:
        raise ValueError("Select either a timestamp column or an elapsed-seconds column.")

    df = pd.DataFrame()
    if id_col:
        df["id"] = raw_df[id_col].astype(str)
    elif "__source_file" in raw_df.columns:
        df["id"] = raw_df["__source_file"].astype(str)
    else:
        df["id"] = "measurement_001"

    df["k30_co2"] = pd.to_numeric(raw_df[co2_col], errors="coerce")

    if timestamp_col:
        df["datetime"] = pd.to_datetime(raw_df[timestamp_col], errors="coerce")
        df["timedelta"] = (
            df.groupby("id")["datetime"]
            .transform(lambda values: (values - values.min()).dt.total_seconds())
            .astype("float")
        )
    else:
        df["timedelta"] = pd.to_numeric(raw_df[elapsed_col], errors="coerce")
        base = pd.Timestamp("2000-01-01")
        offsets = {measurement_id: n for n, measurement_id in enumerate(df["id"].drop_duplicates())}
        df["datetime"] = [
            base + pd.Timedelta(days=offsets[measurement_id]) + pd.Timedelta(seconds=float(seconds))
            if pd.notna(seconds) else pd.NaT
            for measurement_id, seconds in zip(df["id"], df["timedelta"])
        ]

    pressure = _require_or_fill(
        raw_df,
        pressure_col,
        "bmp_pressure",
        fill_missing_environment,
        default_pressure_pa,
    )
    pressure = pd.to_numeric(pressure, errors="coerce")
    if pressure_col and pressure_unit == "kPa":
        pressure = pressure * 1000.0
    df["bmp_pressure"] = pressure

    df["si_temperature"] = pd.to_numeric(
        _require_or_fill(
            raw_df,
            temperature_col,
            "si_temperature",
            fill_missing_environment,
            default_temperature_c,
        ),
        errors="coerce",
    )
    df["si_humidity"] = pd.to_numeric(
        _require_or_fill(
            raw_df,
            humidity_col,
            "si_humidity",
            fill_missing_environment,
            default_humidity_percent,
        ),
        errors="coerce",
    )

    df = df.sort_values(["id", "datetime", "timedelta"]).reset_index(drop=True)
    return df[CANONICAL_COLUMNS]


def validate_fcs_dataframe(df):
    missing = [column for column in CANONICAL_COLUMNS if column not in df.columns]
    null_counts = df[CANONICAL_COLUMNS].isna().sum().to_dict() if not missing else {}
    valid = not missing and all(count == 0 for count in null_counts.values())
    return {
        "valid": valid,
        "missing_columns": missing,
        "null_counts": null_counts,
        "n_measurements": int(df["id"].nunique()) if "id" in df.columns else 0,
        "n_rows": int(len(df)),
    }


def summarize_measurements(df):
    summary = (
        df.groupby("id")
        .agg(
            n_rows=("timedelta", "size"),
            start=("datetime", "min"),
            end=("datetime", "max"),
            duration_s=("timedelta", "max"),
            min_co2=("k30_co2", "min"),
            max_co2=("k30_co2", "max"),
        )
        .reset_index()
    )
    return summary


def plot_measurement(df, measurement_id=None):
    measurement_id = measurement_id or df["id"].iloc[0]
    selected = df[df["id"].astype(str) == str(measurement_id)]
    fig, ax = plt.subplots(figsize=(7, 3.5), dpi=120)
    ax.plot(selected["timedelta"], selected["k30_co2"], color="#1f77b4", linewidth=1.8)
    ax.scatter(selected["timedelta"], selected["k30_co2"], color="#1f77b4", s=10, alpha=0.4)
    ax.set_xlabel("Elapsed time [s]")
    ax.set_ylabel(r"$CO_2$ [ppm]")
    ax.set_title(f"Measurement: {measurement_id}")
    fig.tight_layout()
    return fig


def run_fcs_processing(
    df,
    *,
    chamber_id,
    output_folder=DEFAULT_OUTPUT_DIR,
    area=314.0,
    volume=6283.0,
    use_mcmc=False,
    n_mc=500,
    sensor_precision=None,
):
    validation = validate_fcs_dataframe(df)
    if not validation["valid"]:
        raise ValueError(f"Dataframe is not ready for FCS processing: {validation}")

    output_folder = pathlib.Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)

    processor = Multiprocessor()
    metadata = {"area": area, "volume": volume}
    if use_mcmc:
        return processor.run_MC(
            df=df,
            chamber_id=chamber_id,
            output_folder=str(output_folder),
            save_netcdf=True,
            sensor_precision=sensor_precision,
            n_MC=n_mc,
            metadata=metadata,
        )
    return processor.run(
        df=df,
        chamber_id=chamber_id,
        output_folder=str(output_folder),
        metadata=metadata,
    )


## Configuration


In [ ]:
INPUT_MODE = "JSON folder"  # "JSON folder" or "CSV"
INPUT_PATH = "/Users/alexnaokiasatokobayashi/Downloads/test_folder/2026-04-17/"

CSV_PATTERN = "*.csv"
DELIMITER = ","

# CSV column mapping. Leave optional environmental columns empty only when FILL_MISSING_ENVIRONMENT=True.
ID_COL = ""
TIMESTAMP_COL = "timestamp"
ELAPSED_COL = ""
CO2_COL = "co2"
PRESSURE_COL = "pressure"
PRESSURE_UNIT = "Pa"  # "Pa" or "kPa"
TEMPERATURE_COL = "temp"
HUMIDITY_COL = "rh"
FILL_MISSING_ENVIRONMENT = False
DEFAULT_PRESSURE_PA = 101325.0
DEFAULT_TEMPERATURE_C = 20.0
DEFAULT_HUMIDITY_PERCENT = 70.0

CHAMBER_ID = "analysis"
OUTPUT_FOLDER = DEFAULT_OUTPUT_DIR
AREA_CM2 = 314.0
VOLUME_CM3 = 6283.0
USE_MCMC = False
N_MC = 500
SENSOR_PRECISION = None

RUN_FCS = False


## Load Raw Input


In [ ]:
if INPUT_MODE == "JSON folder":
    json_files, ignored_json_files = find_json_files(INPUT_PATH)
    print(f"Matched {len(json_files)} usable JSON file(s).")
    if ignored_json_files:
        print(f"Ignored {len(ignored_json_files)} small JSON file(s).")
    df = load_json_folder(INPUT_PATH)
    raw = df
elif INPUT_MODE == "CSV":
    csv_files = find_input_files(INPUT_PATH, pattern=CSV_PATTERN)
    print(f"Matched {len(csv_files)} CSV file(s).")
    raw = load_csv_files(INPUT_PATH, pattern=CSV_PATTERN, delimiter=DELIMITER)
    display(raw.head())
else:
    raise ValueError("INPUT_MODE must be 'JSON folder' or 'CSV'.")


## Normalize And Validate


In [ ]:
if INPUT_MODE == "CSV":
    df = normalize_csv_dataframe(
        raw,
        id_col=ID_COL,
        timestamp_col=TIMESTAMP_COL,
        elapsed_col=ELAPSED_COL,
        co2_col=CO2_COL,
        pressure_col=PRESSURE_COL,
        temperature_col=TEMPERATURE_COL,
        humidity_col=HUMIDITY_COL,
        pressure_unit=PRESSURE_UNIT,
        fill_missing_environment=FILL_MISSING_ENVIRONMENT,
        default_pressure_pa=DEFAULT_PRESSURE_PA,
        default_temperature_c=DEFAULT_TEMPERATURE_C,
        default_humidity_percent=DEFAULT_HUMIDITY_PERCENT,
    )

validation = validate_fcs_dataframe(df)
display(df.head())
display(summarize_measurements(df).head())
display(validation)
if not validation["valid"]:
    raise ValueError(f"Dataframe is not ready for FCS processing: {validation}")


## Preview One Measurement


In [ ]:
measurement_id = df["id"].astype(str).iloc[0]
display(plot_measurement(df, measurement_id))


## Run FCS


In [ ]:
if RUN_FCS:
    result = run_fcs_processing(
        df,
        chamber_id=CHAMBER_ID,
        output_folder=OUTPUT_FOLDER,
        area=AREA_CM2,
        volume=VOLUME_CM3,
        use_mcmc=USE_MCMC,
        n_mc=N_MC,
        sensor_precision=SENSOR_PRECISION,
    )
    print("FCS processing complete.")
    print(f"Output folder: {OUTPUT_FOLDER}")
    display(result)
else:
    print("RUN_FCS is False. Set RUN_FCS = True in the configuration cell to write NetCDF outputs.")
